# CS5785 Midterm Project 2026

This notebook includes baseline models and code for preparing the two Kaggle submissions. See the project PDF for the full requirements.

## Load the data

Use the project `.venv/bin/python` kernel. The CSV files are in `part1-data/` and `part2-data/` inside the midterm folder. Run from this folder or the parent workspace root. Baseline submission CSVs are saved in the midterm folder; Part 1 label recovery and comparisons are implemented below; Part 2 error analysis and model improvement remain to be completed.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, log_loss

In [2]:
from pathlib import Path

# Support both the notebook folder and the parent workspace as working directories.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "part1-data").is_dir():
    PROJECT_DIR = PROJECT_DIR / "applied-ml-midterm"
if not all((PROJECT_DIR / folder).is_dir() for folder in ("part1-data", "part2-data")):
    raise FileNotFoundError("Run from the midterm folder or its parent workspace root.")

PART1_DIR = PROJECT_DIR / "part1-data"
PART2_DIR = PROJECT_DIR / "part2-data"

reason_train = pd.read_csv(PART1_DIR / "part1_train.csv")
reason_val = pd.read_csv(PART1_DIR / "part1_val.csv")
reason_test = pd.read_csv(PART1_DIR / "part1_test.csv")

outcome_train = pd.read_csv(PART2_DIR / "outcome_train.csv")
outcome_val = pd.read_csv(PART2_DIR / "outcome_val.csv")
outcome_test = pd.read_csv(PART2_DIR / "outcome_test.csv")

pathway_catalog = pd.read_csv(PART1_DIR / "pathway_catalog.csv")

## Part 1: Stopping reasons

Each trial can have several reason labels. In the training set, `-100` marks missing labels.

### Prepare the data

Use the 150 embedding columns and standardize them using the training data.

In [3]:
labels = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
features = [f"emb_{i}" for i in range(150)]
labeled = (reason_train[labels] != -100).all(axis=1)

scaler = StandardScaler()
X_train = scaler.fit_transform(reason_train[features])
X_val = scaler.transform(reason_val[features])
X_test = scaler.transform(reason_test[features])

y_train = reason_train.loc[labeled, labels]
y_val = reason_val[labels]

print("Labeled trials:", labeled.sum())
print("Unlabeled trials:", (~labeled).sum())

Labeled trials: 157
Unlabeled trials: 1014


### Logistic regression baseline

Train on the labeled trials and evaluate on the validation set.

In [4]:
reason_model = OneVsRestClassifier(
    LogisticRegression(max_iter=500, solver="liblinear", random_state=42)
)
reason_model.fit(X_train[labeled], y_train)
val_pred = reason_model.predict(X_val)

print("Sample-wise F1:", round(f1_score(y_val, val_pred, average="samples", zero_division=0), 4))
print("Macro F1:", round(f1_score(y_val, val_pred, average="macro", zero_division=0), 4))

per_label_f1 = f1_score(y_val, val_pred, average=None, zero_division=0)
pd.Series(per_label_f1, index=labels, name="F1").round(4)

Sample-wise F1: 0.8654
Macro F1: 0.7067


Safety         0.8571
Funding        0.2857
Efficacy       0.8276
Recruitment    0.9796
Other          0.5833
Name: F1, dtype: float64

### Part 1 experiments: K-means and GMM label recovery

Following HW3, cluster **all training embeddings**, including the unlabeled rows. Map each cluster to five independent reason frequencies using only the 157 visible training labels (never validation labels). K-means uses hard membership; GMM uses soft responsibilities. Smooth frequencies toward the labeled-training prior with two pseudo-observations so clusters without labeled members have a defined fallback. Threshold these frequencies to recover all 1,014 missing label vectors. Each stopped trial receives at least one reason (the highest-scoring reason if no threshold is met).

Fit five binary logistic regressions on the augmented training set, giving recovered rows less or equal weight than true labels. Keep known labels unchanged. Compare a fixed grid on validation sample-wise F1; use macro F1 only to break ties. Test embedding-only, added start year, and added text/drug-knowledge features. Clustering always uses only the reason embeddings, so feature ablations isolate the supervised stage. No test labels, IDs, or validation examples enter fitting. The validation set is small, so differences are exploratory rather than statistical evidence of improvement.

In [5]:
import json
import warnings
import numpy as np
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

RESULTS_DIR = PROJECT_DIR / "part1-results"
RESULTS_DIR.mkdir(exist_ok=True)
label_mask = labeled.to_numpy()
y_known = y_train.to_numpy(dtype=int)
y_validation = y_val.to_numpy(dtype=int)
assert label_mask.sum() == 157 and (~label_mask).sum() == 1014
assert reason_train.loc[~labeled, labels].eq(-100).all().all()
assert np.isin(y_known, [0, 1]).all()
assert not set(reason_train.ID) & set(reason_val.ID)
assert not set(reason_train.ID) & set(reason_test.ID)
assert not set(reason_val.ID) & set(reason_test.ID)

label_summary = pd.DataFrame({
    "visible_positive": y_known.sum(axis=0),
    "visible_rate": y_known.mean(axis=0),
    "validation_rate": y_validation.mean(axis=0)
}, index=labels)
print("Training years (visible / masked):",
      reason_train.loc[labeled, "start_year"].mean(),
      reason_train.loc[~labeled, "start_year"].mean())
display(label_summary.round(4))

# All feature transformers learn only from training rows.
year_scaler = StandardScaler()
years = [year_scaler.fit_transform(reason_train[["start_year"]]),
         year_scaler.transform(reason_val[["start_year"]]),
         year_scaler.transform(reason_test[["start_year"]])]
embedding_sets = [sparse.csr_matrix(x) for x in (X_train, X_val, X_test)]
feature_sets = {
    "embeddings": tuple(embedding_sets),
    "embeddings_year": tuple(sparse.hstack([x, year], format="csr")
                             for x, year in zip(embedding_sets, years))
}
reason_text_columns = ["criteria", "diseases_cleaned", "drugs", "icdcodes",
                       "drug_targets", "drug_actions", "drug_pathways"]
def reason_text(frame):
    return frame[reason_text_columns].fillna("").agg(" ".join, axis=1)

reason_vectorizer = TfidfVectorizer(min_df=3, max_features=3000,
                                   ngram_range=(1, 1), sublinear_tf=True)
text_sets = [reason_vectorizer.fit_transform(reason_text(reason_train)),
             reason_vectorizer.transform(reason_text(reason_val)),
             reason_vectorizer.transform(reason_text(reason_test))]
feature_sets["embeddings_year_text"] = tuple(
    sparse.hstack([x, year, text], format="csr")
    for x, year, text in zip(embedding_sets, years, text_sets)
)
print("Feature dimensions:", {name: data[0].shape[1] for name, data in feature_sets.items()})

Training years (visible / masked): 2001.71974522293 2006.424063116371


,visible_positive,visible_rate,validation_rate
Safety,16,0.1019,0.1154
Funding,8,0.0510,0.0513
Efficacy,17,0.1083,0.1795
Recruitment,101,0.6433,0.6410
Other,25,0.1592,0.1282


Feature dimensions: {'embeddings': 150, 'embeddings_year': 151, 'embeddings_year_text': 3151}


In [6]:
def reason_metrics(truth, prediction):
    result = {
        "sample_f1": f1_score(truth, prediction, average="samples", zero_division=0),
        "macro_f1": f1_score(truth, prediction, average="macro", zero_division=0)
    }
    result.update(zip(labels, f1_score(truth, prediction, average=None, zero_division=0)))
    return result


def threshold_reasons(probabilities, threshold=0.5):
    prediction = (probabilities >= threshold).astype(int)
    empty = prediction.sum(axis=1) == 0
    prediction[np.flatnonzero(empty), probabilities[empty].argmax(axis=1)] = 1
    return prediction


def fit_reason_models(x, y, c, weights=None):
    # Explicit binary fits allow per-row weights for pseudo-labels.
    return [LogisticRegression(C=c, max_iter=1000, solver="liblinear", random_state=42)
            .fit(x, y[:, j], sample_weight=weights) for j in range(len(labels))]


def reason_probabilities(models, x):
    return np.column_stack([model.predict_proba(x)[:, 1] for model in models])


# Preserve the starter baseline exactly (including its original 0.5 decisions).
records = [{"method": "starter", "features": "embeddings", "k": 0,
            "pseudo_threshold": 0.5, "pseudo_weight": 0.0, "C": 1.0,
            **reason_metrics(y_validation, val_pred)}]
# Tune supervised controls using exactly the same final prediction rule.
for feature_name, (train_x, val_x, _) in feature_sets.items():
    for c in (0.1, 1.0, 10.0):
        models = fit_reason_models(train_x[label_mask], y_known, c)
        pred = threshold_reasons(reason_probabilities(models, val_x))
        records.append({"method": "supervised", "features": feature_name, "k": 0,
                        "pseudo_threshold": 0.5, "pseudo_weight": 0.0, "C": c,
                        **reason_metrics(y_validation, pred)})

# Cache recovered labels: changing the downstream feature set does not recluster.
pseudo_labels = {}
cluster_diagnostics = []
prior = y_known.mean(axis=0)
for method in ("kmeans", "gmm"):
    for k in (5, 10, 20):
        if method == "kmeans":
            clusterer = KMeans(n_clusters=k, init="k-means++", n_init=10,
                               max_iter=300, tol=1e-4, random_state=42)
        else:
            clusterer = GaussianMixture(n_components=k, covariance_type="diag",
                                        n_init=3, init_params="kmeans", max_iter=200,
                                        tol=1e-3, reg_covar=1e-5, random_state=42)
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            clusterer.fit(X_train)
        if method == "kmeans":
            membership = np.eye(k)[clusterer.labels_]
        else:
            membership = clusterer.predict_proba(X_train)
        known_membership = membership[label_mask]
        mass = known_membership.sum(axis=0)
        cluster_rates = (known_membership.T @ y_known + 2.0 * prior) / (mass[:, None] + 2.0)
        recovered_probabilities = membership[~label_mask] @ cluster_rates
        cluster_diagnostics.append({
            "method": method, "k": k, "iterations": int(clusterer.n_iter_),
            "converged": bool(getattr(clusterer, "converged_", clusterer.n_iter_ < 300)),
            "clusters_with_labeled_mass_below_1": int((mass < 1).sum()),
            "convergence_warnings": sum(issubclass(w.category, ConvergenceWarning) for w in caught)
        })
        for threshold in (0.3, 0.5):
            augmented = np.empty((len(reason_train), len(labels)), dtype=int)
            augmented[label_mask] = y_known
            augmented[~label_mask] = threshold_reasons(recovered_probabilities, threshold)
            assert np.array_equal(augmented[label_mask], y_known)
            pseudo_labels[(method, k, threshold)] = augmented
            for weight in (0.25, 1.0):
                weights = np.where(label_mask, 1.0, weight)
                for c in (0.1, 1.0):
                    models = fit_reason_models(feature_sets["embeddings"][0], augmented, c, weights)
                    pred = threshold_reasons(reason_probabilities(models, feature_sets["embeddings"][1]))
                    records.append({"method": method, "features": "embeddings", "k": k,
                                    "pseudo_threshold": threshold, "pseudo_weight": weight, "C": c,
                                    **reason_metrics(y_validation, pred)})


def ranked_results(rows):
    # Stable order makes exact ties reproducible; sample-wise F1 is primary.
    return pd.DataFrame(rows).sort_values(["sample_f1", "macro_f1"], ascending=False, kind="stable")

# Feature ablations reuse each method's best embedding-only recovery settings.
for method in ("kmeans", "gmm"):
    best = ranked_results([r for r in records if r["method"] == method]).iloc[0]
    augmented = pseudo_labels[(method, int(best.k), float(best.pseudo_threshold))]
    weights = np.where(label_mask, 1.0, best.pseudo_weight)
    for feature_name in ("embeddings_year", "embeddings_year_text"):
        train_x, val_x, _ = feature_sets[feature_name]
        for c in (0.1, 1.0, 10.0):
            models = fit_reason_models(train_x, augmented, c, weights)
            pred = threshold_reasons(reason_probabilities(models, val_x))
            records.append({"method": method, "features": feature_name, "k": int(best.k),
                            "pseudo_threshold": best.pseudo_threshold,
                            "pseudo_weight": best.pseudo_weight, "C": c,
                            **reason_metrics(y_validation, pred)})

experiments = ranked_results(records).reset_index(drop=True)
experiments.to_csv(RESULTS_DIR / "experiments.csv", index=False)
diagnostics = pd.DataFrame(cluster_diagnostics)
diagnostics.to_csv(RESULTS_DIR / "cluster_diagnostics.csv", index=False)
comparison = experiments.groupby(["method", "features"], sort=False).head(1)
display(comparison.round(4))
display(diagnostics)
print("Validation configurations evaluated:", len(experiments))

,method,features,k,pseudo_threshold,pseudo_weight,C,sample_f1,macro_f1,Safety,Funding,Efficacy,Recruitment,Other
0,supervised,embeddings,0,0.5,0.00,1.0,0.8782,0.6990,0.8571,0.2857,0.8125,0.9796,0.5600
2,starter,embeddings,0,0.5,0.00,1.0,0.8654,0.7067,0.8571,0.2857,0.8276,0.9796,0.5833
3,kmeans,embeddings,10,0.3,0.25,0.1,0.8547,0.6923,0.8235,0.4000,0.7857,0.9524,0.5000
6,kmeans,embeddings_year,10,0.3,0.25,0.1,0.8526,0.6869,0.8235,0.4000,0.7586,0.9524,0.5000
7,kmeans,embeddings_year_text,10,0.3,0.25,0.1,0.8526,0.6869,0.8235,0.4000,0.7586,0.9524,0.5000
13,gmm,embeddings,20,0.3,0.25,0.1,0.8312,0.6418,0.7619,0.4000,0.5263,0.9608,0.5600
19,gmm,embeddings_year,20,0.3,0.25,0.1,0.8226,0.6185,0.7273,0.4000,0.4444,0.9608,0.5600
20,gmm,embeddings_year_text,20,0.3,0.25,0.1,0.8226,0.6185,0.7273,0.4000,0.4444,0.9608,0.5600
42,supervised,embeddings_year_text,0,0.5,0.00,1.0,0.7765,0.5924,0.6154,0.2857,0.7222,0.9796,0.3590
49,supervised,embeddings_year,0,0.5,0.00,10.0,0.7658,0.5795,0.6154,0.2500,0.7027,0.9796,0.3500


,method,k,iterations,converged,clusters_with_labeled_mass_below_1,convergence_warnings
0,kmeans,5,30,True,0,0
1,kmeans,10,19,True,0,0
2,kmeans,20,19,True,0,0
3,gmm,5,17,True,0,0
4,gmm,10,23,True,0,0
5,gmm,20,51,True,0,0


Validation configurations evaluated: 70


### Select and audit the semi-supervised model

Choose the best **K-means/GMM augmented classifier** by validation sample-wise F1, using macro F1 to break ties. Retain the strongest supervised control for an honest comparison even if augmentation is worse. Do not refit on validation labels: those labels remain available for later Part 2 error analysis. Export recovered training labels, validation predictions, and test predictions with explicit provenance so true labels and estimates can be distinguished when joining on trial ID.

The test CSV is used only for applying fitted transformations and generating predictions. No Kaggle score is used for model selection.

In [7]:
selected = experiments[experiments.method.isin(["kmeans", "gmm"])].iloc[0]
selected_y = pseudo_labels[(selected.method, int(selected.k), float(selected.pseudo_threshold))]
selected_train_x, selected_val_x, selected_test_x = feature_sets[selected.features]
selected_models = fit_reason_models(
    selected_train_x, selected_y, float(selected.C),
    np.where(label_mask, 1.0, float(selected.pseudo_weight))
)
selected_val_pred = threshold_reasons(reason_probabilities(selected_models, selected_val_x))
selected_test_pred = threshold_reasons(reason_probabilities(selected_models, selected_test_x))
assert np.isclose(reason_metrics(y_validation, selected_val_pred)["sample_f1"], selected.sample_f1)
assert np.isin(selected_y, [0, 1]).all()
assert np.array_equal(selected_y[label_mask], y_known)

recovered_training = reason_train[["ID"]].copy()
recovered_training[labels] = selected_y
recovered_training["label_source"] = np.where(label_mask, "observed_train", "pseudo_train")
recovered_training.to_csv(RESULTS_DIR / "recovered_training_labels.csv", index=False)
validation_predictions = reason_val[["ID"]].copy()
validation_predictions[labels] = selected_val_pred
validation_predictions["label_source"] = "predicted_validation"
validation_predictions.to_csv(RESULTS_DIR / "validation_predictions.csv", index=False)
test_predictions = reason_test[["ID"]].copy()
test_predictions[labels] = selected_test_pred
test_predictions["label_source"] = "predicted_test"
test_predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
# A ready-to-join reason table for Part 2: use true validation reasons when available.
observed_validation = reason_val[["ID", *labels]].copy()
observed_validation["label_source"] = "observed_validation"
reason_lookup = pd.concat([recovered_training, observed_validation, test_predictions], ignore_index=True)
assert reason_lookup.ID.is_unique
reason_lookup.to_csv(RESULTS_DIR / "reason_lookup.csv", index=False)

label_summary["recovered_positive"] = selected_y[~label_mask].sum(axis=0)
label_summary["recovered_rate"] = selected_y[~label_mask].mean(axis=0)
label_summary.to_csv(RESULTS_DIR / "label_summary.csv", index_label="reason")
selection_info = {name: selected[name] for name in
                  ("method", "features", "k", "pseudo_threshold", "pseudo_weight", "C",
                   "sample_f1", "macro_f1")}
selection_info["k"] = int(selection_info["k"])
selection_info["random_state"] = 42
selection_info["validation_configurations"] = len(experiments)
(RESULTS_DIR / "selected_model.json").write_text(json.dumps(selection_info, indent=2) + "\n")
print("Selected semi-supervised configuration:", selection_info)
display(label_summary.round(4))

# Generate the report table directly from measured results.
baseline = experiments[experiments.method == "starter"].iloc[0]
best_supervised = experiments[experiments.method == "supervised"].iloc[0]
best_kmeans = experiments[experiments.method == "kmeans"].iloc[0]
best_gmm = experiments[experiments.method == "gmm"].iloc[0]
table_rows = [r"\begin{tabular}{lrrrr}", r"\toprule",
              'Metric & Starter & Sup. & K-means & GMM \\\\', r"\midrule"]
for metric, title in [("sample_f1", "Sample F1"), ("macro_f1", "Macro F1"), *[(x, x) for x in labels]]:
    values = [row[metric] for row in (baseline, best_supervised, best_kmeans, best_gmm)]
    table_rows.append(title + " & " + " & ".join(f"{value:.4f}" for value in values) + ' \\\\')
table_rows.extend([r"\bottomrule", r"\end{tabular}"])
(PROJECT_DIR / "writeup" / "part1_results.tex").write_text("\n".join(table_rows) + "\n")

feature_rows = [r"\begin{tabular}{lrrr}", r"\toprule",
                "Features & Supervised & K-means & GMM " + chr(92) * 2, r"\midrule"]
for name, title in [("embeddings", "E"), ("embeddings_year", "E+Y"),
                    ("embeddings_year_text", "E+Y+T")]:
    values = [experiments[(experiments.method == method) & (experiments.features == name)].iloc[0].sample_f1
              for method in ("supervised", "kmeans", "gmm")]
    feature_rows.append(title + " & " + " & ".join(f"{value:.4f}" for value in values) + " " + chr(92) * 2)
feature_rows.extend([r"\bottomrule", r"\end{tabular}"])
(PROJECT_DIR / "writeup" / "part1_feature_results.tex").write_text("\n".join(feature_rows) + "\n")

import platform
from importlib.metadata import version
environment = {name: version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "nbclient"]}
environment["python"] = platform.python_version()
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")

Selected semi-supervised configuration: {'method': 'kmeans', 'features': 'embeddings', 'k': 10, 'pseudo_threshold': np.float64(0.3), 'pseudo_weight': np.float64(0.25), 'C': np.float64(0.1), 'sample_f1': np.float64(0.8547008547008546), 'macro_f1': np.float64(0.6923249299719888), 'random_state': 42, 'validation_configurations': 70}


,visible_positive,visible_rate,validation_rate,recovered_positive,recovered_rate
Safety,16,0.1019,0.1154,98,0.0966
Funding,8,0.0510,0.0513,0,0.0000
Efficacy,17,0.1083,0.1795,128,0.1262
Recruitment,101,0.6433,0.6410,749,0.7387
Other,25,0.1592,0.1282,137,0.1351


138

### Save Part 1 predictions

Save the selected semi-supervised model's test predictions as `ID,Predicted`. Reason names are space-separated. These are local files; this notebook does not upload them to Kaggle.

In [8]:
test_pred = selected_test_pred
predicted_reasons = []

for row in test_pred:
    names = [label for label, value in zip(labels, row) if value == 1]
    predicted_reasons.append(" ".join(names))

part1_submission = reason_test[["ID"]].copy()
part1_submission["Predicted"] = predicted_reasons
part1_submission.to_csv(PROJECT_DIR / "part1_submission.csv", index=False)
part1_submission.head()

assert part1_submission.ID.equals(reason_test.ID)
assert part1_submission.Predicted.str.len().gt(0).all()
assert len(part1_submission) == 312

## Part 2: Trial outcomes

Predict whether a trial met its primary endpoint (`label=1`). Fit a separate model for each phase.

### Prepare the text

Combine the eligibility criteria, diseases, drugs, and ICD codes.

In [9]:
text_columns = ["criteria", "diseases_cleaned", "drugs", "icdcodes"]

def trial_text(df):
    return df[text_columns].fillna("").agg(" ".join, axis=1)

### Logistic regression baselines

Fit a bag-of-words model for each phase and report the validation metrics.

In [10]:
val_results = outcome_val[["ID", "trial_id", "phase", "label"]].copy()
part2_submission = outcome_test[["ID"]].copy()
scores = []

for phase in ["I", "II", "III"]:
    train = outcome_train[outcome_train["phase"] == phase]
    val = outcome_val[outcome_val["phase"] == phase]
    test = outcome_test[outcome_test["phase"] == phase]

    vectorizer = CountVectorizer(binary=True, min_df=3, max_features=20000)
    X_train_phase = vectorizer.fit_transform(trial_text(train))
    X_val_phase = vectorizer.transform(trial_text(val))
    X_test_phase = vectorizer.transform(trial_text(test))

    model = LogisticRegression(max_iter=500, solver="liblinear", random_state=42)
    model.fit(X_train_phase, train["label"])

    prediction = model.predict(X_val_phase)
    probability = model.predict_proba(X_val_phase)[:, 1]
    val_results.loc[val.index, "prediction"] = prediction
    val_results.loc[val.index, "probability"] = probability
    part2_submission.loc[test.index, "label"] = model.predict_proba(X_test_phase)[:, 1]

    scores.append({
        "Phase": phase,
        "F1": f1_score(val["label"], prediction, zero_division=0),
        "ROC-AUC": roc_auc_score(val["label"], probability),
        "PR-AUC": average_precision_score(val["label"], probability),
        "Log Loss": log_loss(val["label"], probability)
    })

pd.DataFrame(scores).round(4)

,Phase,F1,ROC-AUC,PR-AUC,Log Loss
0,I,0.7006,0.7029,0.7543,0.8791
1,II,0.6141,0.6633,0.6383,1.0603
2,III,0.7739,0.7049,0.8152,0.8808


### Error analysis

Use your Part 1 labels and predictions to group unsuccessful validation trials by reason, matching `trial_id` to `ID`. For each phase and reason group, report the trial count and the fraction predicted successful. Put unmatched trials in a separate “no recorded stopping reason” group, and state which labels were predicted.

In [11]:
failed_val = val_results[val_results["label"] == 0].copy()

# Add your reason-group analysis here.

### Model improvement

Try at least one change to your outcome models. Compare the validation metrics and reason-group error rates with the baseline.

In [12]:
# Add your revised models and comparisons here.

### Save Part 2 predictions

Save success probabilities as `ID,label`. Kaggle evaluates this file using log loss. The predictions below are from the baseline models.

In [13]:
part2_submission.to_csv(PROJECT_DIR / "part2_submission.csv", index=False)
part2_submission.head()

,ID,label
0,TR11252_III,0.140164
1,TR08672_II,0.786970
2,TR07815_I,0.917404
3,TR08013_I,0.998323
4,TR05100_II,0.855623
